In [ ]:
REPO_COMMIT = '4092eb48665076b0af888db462d15485bd5b677a'
SOURCE_DIGEST = None
RUN_TAG = 'p1_kernel_ablation_s01'
JOBS_GPU0 = ['vna__s0','kernel_self__s0','vna_rand__s0']
JOBS_GPU1 = ['s1_replay','vna__s1','kernel_self__s1','vna_rand__s1']
JOBS_CPU = []
INPUTS = {0: {}, 1: {}}  # optional checkpoint, original_run, baseline_checkpoint paths
import time
SESSION_HOURS = 11.5
SESSION_START = time.time()
VNA_DEADLINE = SESSION_START + SESSION_HOURS*3600
ALLOW_OVER_BUDGET = True
DISK_CAP_BYTES = 20_000_000_000


# P1 fixed-checkpoint kernel ablation: s0/s1
Enable T4x2, Internet and github_token. Attach CIFAR-100 and original P1 noBN Inc5
outputs with original full protocol.json, epochs.json and interventions.json.
Baseline end-task1/native-task2-epoch0 checkpoints are reused when available;
otherwise each lane reconstructs baseline task1 (100 epochs) on its historical
source: s0=29cc735, s1=6be6543. All 100 losses/accuracies/LRs must match original
epochs exactly. The s1 missing-record exception below defers proof to its anchor.
The notebook locates compatible
expanded files or restored output archives; use INPUTS paths to resolve ambiguity.
Reference outputs are restored separately per source; only this source pin/run
tag is merged into the active resume directory. Conflicting resumes still refuse.
No task-1 GIR amendments or new preflight. Preserve the original
6be6543 certificate/Gram trust region; actual update norms are observations only.
Radius violations block interpretation. Native K is an untruncated control, not
a rank-matched arm; r=16 counts incoming directions per site for virtual kernels.
Random-A has one seeded trajectory per seed, with no statistical-superiority claim.
Each arm attempts all 8 copied VNA-only steps, then resumes training from step 1
through task 5 with VNA–GIR and the same projection kernel. Independent losses use
a fixed balanced official-test batch; they never select/accept steps. GPU1 audits
the original s1 replay through task2 epoch43 before computing its ablations.
Replay conclusion_allowed depends only on differences; original GPU flags are
unrecorded and current flags are logged. If s1 protocol/epochs remain missing
(including the attached nlnk1607/adadad/vna4_main (2) output), skip replay and
require it2001 L_before=2.853116586477115, KD_before=1.2374861653019964 exactly.
Those results carry interpretation_scope=no_replay_audit; historical task1 trace
equality is not claimed without the original epochs. Seeds 2–7 are held out.
Runtime is unmeasured; multiple sessions may be necessary. Resume with this Output,
the same source pin and original inputs. No source ZIP is required.


In [ ]:
import hashlib, json, os, shutil, subprocess, sys, threading, time, zipfile
os.environ.update(TQDM_DISABLE="1", PYTHONUNBUFFERED="1")
from pathlib import Path
from kaggle_secrets import UserSecretsClient

MAIN_URL = "https://github.com/duyh80456-code/Counterfactual-Projection.git"
REPO = Path("/kaggle/working/counterfactual-projection")
RUN_ROOT = Path("/kaggle/working/p1_kernel_ablation")
OUTPUT = RUN_ROOT

def private_clone(url, destination, branch):
    token = UserSecretsClient().get_secret("github_token").strip()
    if not token: raise RuntimeError("Kaggle Secret github_token is unavailable")
    askpass = Path("/kaggle/working/.counterfactual_git_askpass.py")
    askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update(GITHUB_TOKEN_RUNTIME=token, GIT_ASKPASS=str(askpass),
               GIT_TERMINAL_PROMPT="0")
    try:
        if len(branch) == 40 and all(c in "0123456789abcdef" for c in branch):
            subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                            url, str(destination)], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "fetch", "origin", branch], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "checkout", "--detach", branch], env=env, check=True)
        else:
            subprocess.run(["git", "clone", "--branch", branch, "--single-branch",
                            url, str(destination)], env=env, check=True)
    finally:
        askpass.unlink(missing_ok=True)

for checkout in (REPO,):
    if checkout.exists(): shutil.rmtree(checkout)
OUTPUT.mkdir(parents=True, exist_ok=True)
private_clone(MAIN_URL, REPO, REPO_COMMIT)
print("main:", subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())

if subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() != REPO_COMMIT:
    raise RuntimeError("Pinned repository revision mismatch")

from datetime import datetime, timezone
CODE_COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Resolved code commit:", CODE_COMMIT, "pin:", REPO_COMMIT)
commit_path = OUTPUT / "code_commit.json"
sessions = json.loads(commit_path.read_text()) if commit_path.exists() else []
sessions.append({"timestamp": datetime.now(timezone.utc).isoformat(), "pin": REPO_COMMIT, "commit": CODE_COMMIT})
temporary = commit_path.with_suffix(".json.tmp")
temporary.write_text(json.dumps(sessions, indent=2) + "\n")
temporary.replace(commit_path)

if SOURCE_DIGEST:
    bundles = [p for p in Path('/kaggle/input').rglob('vna_v3_source.zip')
               if hashlib.sha256(p.read_bytes()).hexdigest() == SOURCE_DIGEST]
    if not bundles: raise FileNotFoundError('Attach vna_v3_source.zip matching SOURCE_DIGEST; this notebook uses uncommitted source')
    with zipfile.ZipFile(bundles[0]) as archive:
        for name in archive.namelist():
            target = (REPO/name).resolve()
            if not target.is_relative_to(REPO.resolve()): raise ValueError('unsafe source bundle member')
        archive.extractall(REPO)
    os.environ['VNA_SOURCE_DIGEST'] = SOURCE_DIGEST
    shutil.copy2(bundles[0], RUN_ROOT/'vna_v3_source.zip')
    (RUN_ROOT/'source_manifest.json').write_text(json.dumps(dict(code_commit=REPO_COMMIT, source_digest=SOURCE_DIGEST)))

os.environ.pop('VNA_SOURCE_DIGEST',None)


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO) + "[test,vision,diagnostics]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements-upstream.txt")], check=True)
subprocess.run(["bash", "scripts/fetch_third_party.sh"], cwd=REPO, check=True)
RUNTIME_PYTHONPATH = os.pathsep.join(filter(None, (
    str(REPO), os.environ.get("PYTHONPATH", ""))))
test_env = os.environ.copy(); test_env.update(PYTHONUNBUFFERED="1", TQDM_DISABLE="1", PYTHONPATH=RUNTIME_PYTHONPATH, REQUIRE_UPSTREAM="1", PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION="python")
DATA_ROOT = Path('/kaggle/input')
MNIST_ROOT = next((p.parent for p in Path('/kaggle/input').rglob('mnist_') if p.is_file()),None)
if JOBS_GPU0 or JOBS_GPU1:
    import torch
    if torch.cuda.device_count() != 2:
        raise RuntimeError(f"Select T4 x2; found {torch.cuda.device_count()} GPU(s)")
    required_cifar_files = {"train", "test", "meta"}
    cifar_dirs = []
    for directory, _, filenames in os.walk("/kaggle/input", followlinks=True):
        path = Path(directory)
        if (path.name == "cifar-100-python" and
                required_cifar_files.issubset(filenames)):
            cifar_dirs.append(path.parent)
    cifar_dirs = sorted(set(cifar_dirs), key=str)
    if not cifar_dirs:
        visible = []
        for directory, dirnames, filenames in os.walk(
                "/kaggle/input", followlinks=True):
            visible.append({
                "directory": directory,
                "directories": sorted(dirnames)[:10],
                "files": sorted(filenames)[:10],
            })
            if len(visible) == 20:
                break
        raise FileNotFoundError(
            "Attach CIFAR-100 containing cifar-100-python/{train,test,meta}. "
            f"Visible input inventory: {visible}")
    DATA_ROOT = cifar_dirs[0]
    print("CIFAR-100 root:", DATA_ROOT)
    
if not (REPO / "experiments/vna_kaggle.py").exists():
    raise RuntimeError("Required module experiments/vna_kaggle.py is missing from the pinned commit")
for module in ("experiments/vna_provenance.py", "methods/vna.py"):
    if not (REPO / module).exists(): raise RuntimeError(f"Required module {module} is missing from the pinned commit")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "matplotlib"], check=True)
sys.path[:0] = [str(REPO)]



subprocess.run([sys.executable, "-m", "pip", "install", "-q", "quadprog"], check=True)


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from experiments.vna_kaggle import restore_stage_archives
from experiments.vna_kaggle import archive_outputs as _archive_outputs
STAGE_B = RUN_ROOT / "outputs"
STAGE_B.mkdir(parents=True, exist_ok=True)
REFERENCE_INPUTS = RUN_ROOT/'reference_inputs'
restore_stage_archives(Path('/kaggle/input'), REFERENCE_INPUTS, 'outputs', namespace=True)
restore_stage_archives(Path('/kaggle/input'), STAGE_B, 'outputs', prefix=REPO_COMMIT[:12]+'/'+RUN_TAG)
ENV = {**test_env, "PYTHONUNBUFFERED": "1", "TQDM_DISABLE": "1", "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2",
       "PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION": "python", "VNA_DEADLINE": str(VNA_DEADLINE)}
session_paused = threading.Event()
lane_state = {gpu: {"job": "idle", "last": ""} for gpu in range(2)}
def pause_due():
    if time.time() >= VNA_DEADLINE - 600: session_paused.set()
    return session_paused.is_set()
def launch(cmd, gpu, name, log_path):
    from experiments.vna_resume import launch_logged
    return launch_logged(cmd, cwd=REPO, env=gpu_env(gpu), log_path=log_path,
                         lane=gpu, job=name, lanes=lane_state,
                         paused=session_paused, deadline=VNA_DEADLINE)
def heartbeat():
    from experiments.vna_resume import print_heartbeat
    while not archive_stop.wait(600):
        print_heartbeat(RUN_ROOT / "outputs", SESSION_START, VNA_DEADLINE, lane_state)
archive_lock = threading.Lock(); archive_stop = threading.Event(); archive_errors = []
def archive_outputs():
    with archive_lock:
        return _archive_outputs(RUN_ROOT, budget=DISK_CAP_BYTES,
                                folders=("outputs",), exclude_parts=("data",))
def periodic_archive():
    while not archive_stop.wait(3600):
        try: archive_outputs()
        except Exception as error:
            archive_errors.append(str(error)); print("Archive error:", error, flush=True)
def gpu_env(gpu): return {**ENV, "CUDA_VISIBLE_DEVICES": str(gpu)}
archive_errors = []
lane_timings = {}
archiver = threading.Thread(target=periodic_archive, daemon=True); archiver.start()
heartbeat_thread = threading.Thread(target=heartbeat, daemon=True); heartbeat_thread.start()


In [ ]:
from diagnostics.kernel_ablation import discover_inputs, TASK1_COMMITS
from experiments.runners.pycil import upstream_imports
upstream_imports()
ROOT=STAGE_B/REPO_COMMIT[:12]/RUN_TAG
ROOT.mkdir(parents=True,exist_ok=True)
ENV.update(OMP_NUM_THREADS='1',MKL_NUM_THREADS='1')
RESOLVED={seed:discover_inputs((Path('/kaggle/input'),REFERENCE_INPUTS,STAGE_B),seed,INPUTS[seed]) for seed in (0,1)}
print('INPUTS',json.dumps(RESOLVED,indent=2),flush=True)
for seed in (0,1):
    if not RESOLVED[seed].get('checkpoint'):
        historical=REPO.parent/('kernel_source_'+TASK1_COMMITS[seed][:12])
        if not historical.exists():private_clone(MAIN_URL,historical,TASK1_COMMITS[seed])

def command(seed,arm,mode,out):
    cmd=[sys.executable,'-u','-m','diagnostics.kernel_ablation','--seed',str(seed),'--arm',arm,
        '--mode',mode,'--original-run',RESOLVED[seed]['original_run'],
        '--anchor-dir',str(ROOT/'anchors'),'--data-dir',str(DATA_ROOT),'--out',str(out),'--device','cuda:0']
    if RESOLVED[seed].get('checkpoint'):cmd+=['--checkpoint',RESOLVED[seed]['checkpoint']]
    if seed==1:cmd+=['--protocol-template',str(Path(RESOLVED[0]['original_run'])/'protocol.json')]
    if RESOLVED[seed].get('baseline_checkpoint'):cmd+=['--baseline-checkpoint',RESOLVED[seed]['baseline_checkpoint']]
    return cmd

def lane(gpu,seed):
    if not RESOLVED[seed].get('checkpoint'):
        if pause_due():return
        name=f'make_task1__s{seed}';folder=ROOT/name
        if not launch(command(seed,'vna','make_task1',folder),gpu,name,RUN_ROOT/(name+'.log')):return
        RESOLVED[seed]['checkpoint']=str(folder/'end_task1/task1.pt')
        print('TASK1 RECONSTRUCTION',seed,(folder/'task1_verification.json').read_text(),flush=True)
        archive_outputs()
    if seed==1 and RESOLVED[seed]['replay_available']:
        if pause_due():return
        folder=ROOT/'s1_replay'
        if not launch(command(1,'vna','replay',folder),gpu,'s1_replay',RUN_ROOT/'s1_replay.log'):return
        audit=json.loads((folder/'replay.json').read_text())
        print('S1 REPLAY AUDIT',json.dumps(audit),flush=True)
        print('S1 conclusions gated:',not audit.get('conclusion_allowed',False),flush=True)
        archive_outputs()
    elif seed==1:
        print('S1 interpretation_scope=no_replay_audit; exact registered it2001 L/KD required',flush=True)
    for arm in ('vna','kernel_self','vna_rand'):
        if pause_due():return
        name=f'{arm}__s{seed}';folder=ROOT/name
        if not launch(command(seed,arm,'ablation',folder),gpu,name,RUN_ROOT/(name+'.log')):return
        archive_outputs()

try:
    with ThreadPoolExecutor(max_workers=2) as pool:
        for future in (pool.submit(lane,0,0),pool.submit(lane,1,1)):future.result()
finally:
    archive_stop.set();archiver.join(timeout=10);archive_outputs()


In [ ]:
for seed in (0,1):
    if seed==1:
        path=ROOT/'s1_replay'/'replay.json'
        print('S1 AUDIT:',path.read_text() if path.exists() else 'no_replay_audit: see anchor audit and summary scope')
    for arm in ('vna','kernel_self','vna_rand'):
        folder=ROOT/f'{arm}__s{seed}'
        print('\nSEED',seed,'KERNEL',arm)
        if (folder/'chain.json').exists():
            for row in json.loads((folder/'chain.json').read_text()):
                print(json.dumps({k:v for k,v in row.items() if k!='event'}))
        path=folder/'tasks.json'
        if path.exists():
            for task in json.loads(path.read_text()):
                print({k:v for k,v in task.items() if k=='task' or k.endswith(('_old_top1','_new_top1','_average_accuracy'))})
        path=folder/'summary.json'
        print('SUMMARY',path.read_text() if path.exists() else 'PAUSED_OR_PENDING')
archive_outputs()
if archive_errors:raise RuntimeError(archive_errors)
